In [1]:
from api_setup import setup
import json

client, model_id = setup()

In [2]:
def add_user_message(messages, content):
    if isinstance(content, str):
        user_message = {"role": "user", "content": [{"text": content}]}
    else:
        user_message = {"role": "user", "content": content}
    messages.append(user_message)


def add_assistant_message(messages, content):
    if isinstance(content, str):
        assistant_message = {"role": "assistant", "content": [{"text": content}]}
    else:
        assistant_message = {"role": "assistant", "content": content}
    messages.append(assistant_message)


def chat(messages, system=None, tools=None, **kwargs):
    params = {
        "modelId": model_id,
        "messages": messages,
        "inferenceConfig": {
            "temperature": kwargs.get("temperature", 1.0),
            "stopSequences": kwargs.get("stop_sequences", []),
        },
    }

    if system:
        params["system"] = [{"text": system}]

    if tools:
        params["toolConfig"] = {"tools": tools, "toolChoice": {"auto": {}}}

    response = client.converse(**params)
    parts = response["output"]["message"]["content"]

    return {
        "parts": parts,
        "stop_reason": response["stopReason"],
        "text": "\n".join([p["text"] for p in parts if "text" in p])
    }

In [51]:
get_current_datetime_schema = {
    "toolSpec": {
        "name": "get_current_datetime",
        "description": """
            Retrieves the current date time formatted according to the specified format string.
            This tool returns the systems current local date and time as a formatted string.
            It should be used when the user needs to know the current date, time or both in a specific format.
            The tool uses Python's datetime.now() function internally, so it will reflect the servers local
            time zone settings. This tool does not support retrieving dates or times for different time zones
            or historical/future dates.
        """,
        "inputSchema": {
            "json": {
                "type": "object",
                "properties": {
                    "date_format": {
                        "type": "string",
                        "description": """
                            A string that specifies the format of the returned date and time, using Python's strftime format codes. 
                            For example, '%Y-%m-%d' returns only the date as '2025-04-02', '%H:%M:%S' returns only the time as '14:30:45', 
                            and the default '%Y-%m-%d %H:%M:%S' returns both as '2025-04-02 14:30:45'. 
                            Use format codes like %Y (4-digit year), %m (month), %d (day), %H (hour), %M (minute), 
                            and %S (second) to customize the output format.
                        """
                    }
                }
            }
        }
    }
}

add_duration_to_datetime_schema = {
    "toolSpec": {
        "name": "add_duration_to_datetime",
        "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
        "inputSchema": {
            "json": {
                "type": "object",
                "properties": {
                    "datetime_str": {
                        "type": "string",
                        "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.",
                    },
                    "duration": {
                        "type": "number",
                        "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0.",
                    },
                    "unit": {
                        "type": "string",
                        "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
                    },
                    "input_format": {
                        "type": "string",
                        "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'.",
                    },
                },
                "required": ["datetime_str"],
            }
        },
    }
}

set_reminder_schema = {
    "toolSpec": {
        "name": "set_reminder",
        "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
        "inputSchema": {
            "json": {
                "type": "object",
                "properties": {
                    "content": {
                        "type": "string",
                        "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.",
                    },
                    "timestamp": {
                        "type": "string",
                        "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp. The system handles all timezone processing internally, ensuring reminders are triggered at the correct time regardless of where the user is located. Users can simply specify the desired time without worrying about timezone configurations.",
                    },
                },
                "required": ["content", "timestamp"],
            }
        },
    }
}

batch_tool_schema = {
    "toolSpec": {
        "name": "batch_tool",
        "description": "Invoke multiple other tool calls simultaneously",
        "inputSchema": {
            "json": {
                "type": "object",
                "properties": {
                "invocations": {
                    "type": "array",
                    "description": "The tool calls to invoke",
                    "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                        "type": "string",
                        "description": "The name of the tool to invoke"
                        },
                        "arguments": {
                        "type": "string", 
                        "description": "The arguments to the tool, encoded as a JSON string"
                        }
                    },
                    "required": ["name", "arguments"]
                    }
                }
                },
                "required": ["invocations"]
            }
        }
    }
}

In [52]:
# Tool: Return current datetime
from datetime import datetime, timedelta

def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    return datetime.now().strftime(date_format)


In [53]:
# Tool: Returns a future datetime

def add_duration_to_datetime(
        datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)
    kwargs = { unit: duration }
    try:
        if unit == "months":
                month = date.month + duration
                year = date.year + month // 12
                month = month % 12
                if month == 0:
                    month = 12
                    year -= 1
                day = min(
                    date.day,
                    [
                        31,
                        29
                        if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)
                        else 28,
                        31,
                        30,
                        31,
                        30,
                        31,
                        31,
                        30,
                        31,
                        30,
                        31,
                    ][month - 1],
                )
                new_date = date.replace(year=year, month=month, day=day)
        elif unit == "years":
            new_date = date.replace(year=date.year + duration)
        else:
            new_date = date + timedelta(**kwargs)
    except ValueError:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")

In [54]:
# Tool: Creates a reminder (Currently just prints output)
def set_reminder(content, timestamp):
    print(
        f"----\nSetting the following reminder for {timestamp}:\n{content}\n----"
    )

In [ ]:
# def run_tool(tool_name, tool_input):
#     match tool_name:
#         case "get_current_datetime":
#             return get_current_datetime(**tool_input)
#         case "set_reminder":
#             return set_reminder(**tool_input)
#         case "add_duration_to_datetime":
#             return add_duration_to_datetime(**tool_input)
#         case _:
#             raise Exception(f"Unknown tool name: {tool_name}")

In [55]:
# Used for batching multiple tools into one result

def run_batch(tool_input):
    batch_output = []
    for invocation in tool_input["invocations"]:
        tool_name = invocation["name"]
        args = json.loads(invocation["arguments"])
        
        tool_output = run_tool(tool_name, args)
        batch_output.append({"tool_name": tool_name, "output": tool_output})
    
    return batch_output

In [56]:
# Batch tool modification

def run_tool(tool_name, tool_input):
    match tool_name:
        case "get_current_datetime":
            return get_current_datetime(**tool_input)
        case "set_reminder":
            return set_reminder(**tool_input)
        case "add_duration_to_datetime":
            return add_duration_to_datetime(**tool_input)
        case "batch_tool":
            return run_batch(tool_input)
        case _:
            raise Exception(f"Unknown tool name: {tool_name}")

In [57]:
def run_tools(parts):
    tool_requests = [part for part in parts if "toolUse" in part]
    tool_result_parts = []

    for tool_request in tool_requests:
        tool_use_id = tool_request["toolUse"]["toolUseId"]
        tool_name = tool_request["toolUse"]["name"]
        tool_input = tool_request["toolUse"]["input"]

        try:
            tool_output = run_tool(tool_name, tool_input)
            tool_result_part = {
                "toolResult": {
                    "toolUseId": tool_use_id,
                    "content": [{"text": json.dumps(tool_output)}],
                    "status": "success"
                }
            }
        except Exception as e:
            tool_result_part = {
                "toolResult": {
                    "toolUseId": tool_use_id,
                    "content": [{"text": f"Error: {e}"}],
                    "status": "error"
                }
            }

        tool_result_parts.append(tool_result_part)

    return tool_result_parts




In [58]:
def run_conversation(messages):
    while True:
        result = chat(
            messages, 
            tools=[
                get_current_datetime_schema,
                add_duration_to_datetime_schema,
                set_reminder_schema,
                batch_tool_schema
            ]
        )

        add_assistant_message(messages, result["parts"])
        print(result["text"])

        if result["stop_reason"] != "tool_use":
            break

        tool_result_parts = run_tools(result["parts"])
        add_user_message(messages, tool_result_parts)

    return messages

In [10]:
# New Behavior added: Add Duration, Set Reminder

messages = []
add_user_message(messages, "How long is it till Halloween?")
run_conversation(messages)

I'll help you find out how long it is until Halloween!
Based on today's date (October 21, 2026), Halloween is just **10 days away** (October 31st)!


[{'role': 'user', 'content': [{'text': 'How long is it till Halloween?'}]},
 {'role': 'assistant',
  'content': [{'text': "I'll help you find out how long it is until Halloween!"},
   {'toolUse': {'toolUseId': 'tooluse_4v05SPzArGjy9YQDLWP09h',
     'name': 'get_current_datetime',
     'input': {'date_format': '%Y-%m-%d'},
     'type': 'tool_use'}}]},
 {'role': 'user',
  'content': [{'toolResult': {'toolUseId': 'tooluse_4v05SPzArGjy9YQDLWP09h',
     'content': [{'text': '"2026-10-21"'}],
     'status': 'success'}}]},
 {'role': 'assistant',
  'content': [{'text': "Based on today's date (October 21, 2026), Halloween is just **10 days away** (October 31st)!"}]}]

In [11]:
# New Behavior added: Add Duration, Set Reminder

messages = []
add_user_message(messages, "Set a reminder for a week before Halloween next year to find an outfit")
run_conversation(messages)

I'll help you set a reminder for a week before Halloween next year. Let me first get the current date to determine "next year," then calculate the reminder date.
Now I'll calculate the date that is one week before Halloween 2026 (Halloween is October 31st, so one week before is October 24th):
----
Setting the following reminder for 2026-10-24T09:00:00:
Find an outfit for Halloween
----
Perfect! I've set a reminder for **October 24, 2026 at 9:00 AM** to find an outfit for Halloween. This is exactly one week before Halloween (October 31st), giving you plenty of time to prepare your costume!


[{'role': 'user',
  'content': [{'text': 'Set a reminder for a week before Halloween next year to find an outfit'}]},
 {'role': 'assistant',
  'content': [{'text': 'I\'ll help you set a reminder for a week before Halloween next year. Let me first get the current date to determine "next year," then calculate the reminder date.'},
   {'toolUse': {'toolUseId': 'tooluse_jbwucRW0FSL4iuRbzufhM0',
     'name': 'get_current_datetime',
     'input': {'date_format': '%Y'},
     'type': 'tool_use'}}]},
 {'role': 'user',
  'content': [{'toolResult': {'toolUseId': 'tooluse_jbwucRW0FSL4iuRbzufhM0',
     'content': [{'text': '"2026"'}],
     'status': 'success'}}]},
 {'role': 'assistant',
  'content': [{'text': "Now I'll calculate the date that is one week before Halloween 2026 (Halloween is October 31st, so one week before is October 24th):"},
   {'toolUse': {'toolUseId': 'tooluse_V5CHm1eKA3FGz6akndr4j1',
     'name': 'set_reminder',
     'input': {'content': 'Find an outfit for Halloween',
      't

In [59]:
# After Run Conversation

messages = []
add_user_message(messages, "What is March 12th, 2025 + 50 days? Also, what is March 12th, 2025 + 100 days?")
run_conversation(messages)


Here are the results:

- **March 12th, 2025 + 50 days** = **Thursday, May 1st, 2025**
- **March 12th, 2025 + 100 days** = **Friday, June 20th, 2025**


[{'role': 'user',
  'content': [{'text': 'What is March 12th, 2025 + 50 days? Also, what is March 12th, 2025 + 100 days?'}]},
 {'role': 'assistant',
  'content': [{'toolUse': {'toolUseId': 'tooluse_XJ7vXSpf3qbUcMCZpH5HcD',
     'name': 'batch_tool',
     'input': {'invocations': [{'name': 'add_duration_to_datetime',
        'arguments': '{"datetime_str": "2025-03-12", "duration": 50, "unit": "days", "input_format": "%Y-%m-%d"}'},
       {'name': 'add_duration_to_datetime',
        'arguments': '{"datetime_str": "2025-03-12", "duration": 100, "unit": "days", "input_format": "%Y-%m-%d"}'}]},
     'type': 'tool_use'}}]},
 {'role': 'user',
  'content': [{'toolResult': {'toolUseId': 'tooluse_XJ7vXSpf3qbUcMCZpH5HcD',
     'content': [{'text': '[{"tool_name": "add_duration_to_datetime", "output": "Thursday, May 01, 2025 12:00:00 AM"}, {"tool_name": "add_duration_to_datetime", "output": "Friday, June 20, 2025 12:00:00 AM"}]'}],
     'status': 'success'}}]},
 {'role': 'assistant',
  'content': 

In [13]:
# Before Run Conversation

messages = []

add_user_message(messages, "What time is it?")
text, parts = chat(messages, tools=[get_current_datetime_schema])

add_assistant_message(messages, parts)

add_user_message(messages, run_tools(parts))

text, parts = chat(messages, tools=[get_current_datetime_schema])

text


ValueError: too many values to unpack (expected 2)